## Concept 9 — Pipeline Prompts & Composition

### What is it?
Pipeline Prompts let you assemble a final prompt from smaller, independently-defined sub-prompts.
Each piece is testable and reusable. Changing one piece does not break others.

### Pipeline
```
intro_template  (role + company)
     +
task_template   (task + output format)
     +
input_template  (the actual question)
     ↓
PipelinePromptTemplate assembles → final prompt → LLM → Answer
```

### Limitation (what Concept 10 fixes)
Even with good structure, poorly written prompt content gives poor results.
→ Concept 10 covers systematic prompt patterns and quality optimization.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm, TEST_INPUTS, run_inputs
from langchain_core.prompts import PromptTemplate, PipelinePromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm()
print('LLM ready')

### Step 1 — The Problem: One Giant Template
Without composition, large prompts become hard to maintain and impossible to reuse.

In [ ]:
# One giant template — changing the role section risks breaking the task section
monolith = PromptTemplate.from_template(
    'You are a {role} at {company}. '
    'Your task is to {task}. '
    'Always format your response as {output_format}. '
    'Question: {question}'
)

result = llm.invoke(monolith.format(
    role='senior data analyst', company='Acme Corp',
    task='analyze and explain data patterns',
    output_format='3 bullet points',
    question='What are the benefits of data normalization?'
))
print(result.content)

### Step 2 — Define Modular Sub-Prompts
Each sub-prompt has its own variables and can be tested independently.

In [ ]:
# Sub-prompt 1: Who is the LLM?
intro_prompt = PromptTemplate.from_template(
    'You are a {role} at {company}.'
)

# Sub-prompt 2: What should the LLM do and how?
task_prompt = PromptTemplate.from_template(
    'Your task: {task}. Always respond as: {output_format}.'
)

# Sub-prompt 3: The actual user question
question_prompt = PromptTemplate.from_template(
    'Question: {question}'
)

# Test each piece independently
print('Intro:   ', intro_prompt.format(role='data scientist', company='TechCo'))
print('Task:    ', task_prompt.format(task='explain concepts', output_format='3 bullet points'))
print('Question:', question_prompt.format(question='What is overfitting?'))

### Step 3 — Assemble with PipelinePromptTemplate
The final template has slots `{intro}`, `{task}`, `{question_part}` that are filled by sub-prompts.

In [ ]:
# The final template references the OUTPUT of each sub-prompt
final_template = PromptTemplate.from_template(
    '{intro}\n{task}\n{question_part}'
)

pipeline_prompt = PipelinePromptTemplate(
    final_prompt=final_template,
    pipeline_prompts=[
        ('intro',         intro_prompt),
        ('task',          task_prompt),
        ('question_part', question_prompt),
    ]
)

# All variables from all sub-prompts are now top-level inputs
print(f'All input variables: {pipeline_prompt.input_variables}')

### Step 4 — Use the Pipeline Prompt

In [ ]:
final_str = pipeline_prompt.format(
    role='machine learning engineer',
    company='AI Startup',
    task='explain ML concepts clearly',
    output_format='numbered list with one-line explanations',
    question='What are the steps in a machine learning pipeline?'
)
print('Assembled prompt:')
print(final_str)
print('\nLLM Response:')
print(llm.invoke(final_str).content)

### Bonus — Swap One Piece Without Breaking Others
This is the core value: modify the task piece, intro stays the same.

In [ ]:
# New task piece — stricter, code-only output
code_task_prompt = PromptTemplate.from_template(
    'Your task: {task}. Output ONLY Python code, no explanations.'
)

code_pipeline = PipelinePromptTemplate(
    final_prompt=final_template,
    pipeline_prompts=[
        ('intro',         intro_prompt),        # same intro
        ('task',          code_task_prompt),     # swapped task
        ('question_part', question_prompt),      # same question
    ]
)

code_chain = code_pipeline | llm | StrOutputParser()
result = code_chain.invoke({
    'role':          'Python developer',
    'company':       'CodeCo',
    'task':          'write a solution',
    'output_format': 'Python code',   # unused by code_task_prompt but harmless
    'question':      'How do I sort a list of dicts by a key?'
})
print(result)

### Full Function

In [ ]:
explain_chain = pipeline_prompt | llm | StrOutputParser()

def pipeline_prompt_fn(query):
    return explain_chain.invoke({
        'role':          'technical educator',
        'company':       'LearnCo',
        'task':          'explain technical concepts clearly',
        'output_format': '2-3 sentences',
        'question':      query,
    })

run_inputs(pipeline_prompt_fn, label='Concept 9 — Pipeline Prompts')